# Validación de la anotación KNN: Response_AtezoBev_HCC

A diferencia del crossval del ICB_HCC (donde había anotaciones originales de tipo celular
con las que comparar el KNN), este dataset no tiene columna de tipo celular en el paper.
La metadata disponible es clínica: Patient_ID, Response, Treatment y Week.

Por tanto, la validación aquí es de dos tipos:
1. **Validación técnica**: confianza del KNN y distribución de tipos celulares
2. **Validación biológica**: si la composición celular predicha por KNN tiene sentido
   respecto a la variable clínica más importante → Responders vs Non-responders

In [1]:
# Imports

import os
import scanpy as sc
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

sc.settings.verbosity = 1

In [2]:
# Rutas

DATA_DIR    = "/beegfs/home/iruizdealda/HCC_singlecell_project/data"
FIGURES_DIR = "/beegfs/home/iruizdealda/HCC_singlecell_project/figures/crossval_AtezoBev"

os.makedirs(FIGURES_DIR, exist_ok=True)

In [3]:
# Cargar el dataset anotado por KNN
# (tras haber ejecutado annotate_datasets.py)

adata = sc.read_h5ad(f"{DATA_DIR}/adata_Response_AtezoBev_HCC_raw.h5ad")

print(adata)
print("\nColumnas en obs:")
print(adata.obs.columns.tolist())

AnnData object with n_obs × n_vars = 97947 × 36601
    obs: 'Tumour_ID', 'Patient_ID', 'Response', 'Treatment', 'Week', 'batch', 'celltype', 'knn_confidence'
    var: 'highly_variable', 'means', 'dispersions', 'dispersions_norm'
    uns: 'celltype_colors', 'hvg', 'log1p', 'neighbors', 'pca', 'umap'
    obsm: 'X_pca', 'X_umap'
    varm: 'PCs'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

Columnas en obs:
['Tumour_ID', 'Patient_ID', 'Response', 'Treatment', 'Week', 'batch', 'celltype', 'knn_confidence']


In [4]:
# ── 1. VALIDACIÓN TÉCNICA ────────────────────────────────────
# Distribución de tipos celulares predichos por KNN

print("Distribución de tipos celulares (KNN):")
print(adata.obs["celltype"].value_counts())

print(f"\nConfianza media del KNN: {adata.obs['knn_confidence'].mean():.3f}")
cells_low = (adata.obs["knn_confidence"] < 0.5).sum()
print(f"Células con confianza < 0.5: {cells_low} ({cells_low/adata.n_obs*100:.1f}%)")

Distribución de tipos celulares (KNN):
celltype
Hepatocyte     43899
T/NK           28790
Myeloid        11249
Endothelial     7538
B               3663
Fibroblast      2808
Name: count, dtype: int64

Confianza media del KNN: 0.943
Células con confianza < 0.5: 327 (0.3%)


In [5]:
# Distribución clínica

print("Response:")
print(adata.obs["Response"].value_counts())

print("\nWeek:")
print(adata.obs["Week"].value_counts())

print("\nPatient_ID únicos:", adata.obs["Patient_ID"].nunique())

Response:
Response
Responder             61947
NonResponder          22444
DeathBeforeImaging     7822
Name: count, dtype: int64

Week:
Week
W0    97947
Name: count, dtype: int64

Patient_ID únicos: 38


In [6]:
# ── 2. UMAP coloreado por celltype y variables clínicas ─────
# Preprocesar para UMAP si no está ya calculado

if "X_umap" not in adata.obsm:
    print("Calculando UMAP...")
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)
    sc.tl.pca(adata, n_comps=50, use_highly_variable=True)
    sc.pp.neighbors(adata, n_neighbors=15, use_rep="X_pca")
    sc.tl.umap(adata)
else:
    print("UMAP ya calculado, usando el existente.")

sc.settings.figdir = FIGURES_DIR

# UMAP por tipo celular KNN y confianza
sc.pl.umap(
    adata,
    color=["celltype", "knn_confidence"],
    ncols=2,
    title=["Tipo celular (KNN)", "Confianza KNN"],
    save="_AtezoBev_celltype_confidence.png",
    show=False
)
plt.close("all")
print(f"UMAP guardado en: {FIGURES_DIR}")

UMAP ya calculado, usando el existente.
UMAP guardado en: /beegfs/home/iruizdealda/HCC_singlecell_project/figures/crossval_AtezoBev


In [7]:
# UMAP coloreado por variables clínicas

sc.pl.umap(
    adata,
    color=["Response", "Week", "Patient_ID"],
    ncols=3,
    title=["Response", "Week", "Patient_ID"],
    save="_AtezoBev_clinical.png",
    show=False
)
plt.close("all")
print(f"UMAP clínico guardado en: {FIGURES_DIR}")

UMAP clínico guardado en: /beegfs/home/iruizdealda/HCC_singlecell_project/figures/crossval_AtezoBev


In [9]:
# ── 3. VALIDACIÓN BIOLÓGICA ──────────────────────────────────
# Composición celular Responders vs Non-responders
#
# Si el KNN funciona bien, esperamos ver diferencias en la proporción
# de tipos celulares entre Responders y Non-responders, coherentes
# con lo que describe el paper (más CD8 TEMRA en Responders,
# más macrófagos CXCL10+ en Responders, etc.)

# Calcular proporciones por Response
prop = (
    adata.obs
    .groupby(["Response", "celltype"], observed=True)
    .size()
    .reset_index(name="n")
)
prop["total"] = prop.groupby("Response")["n"].transform("sum")
prop["proportion"] = prop["n"] / prop["total"]

print("Composición celular por Response:")
print(prop.pivot(index="celltype", columns="Response", values="proportion").round(3))

Composición celular por Response:
Response     DeathBeforeImaging  NonResponder  Responder
celltype                                                
B                         0.015         0.033      0.045
Endothelial               0.050         0.103      0.067
Fibroblast                0.027         0.041      0.026
Hepatocyte                0.673         0.344      0.461
Myeloid                   0.058         0.115      0.123
T/NK                      0.177         0.364      0.279


/tmp/ipykernel_4115021/2077581718.py:16: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  prop["total"] = prop.groupby("Response")["n"].transform("sum")


In [10]:
# Barplot de composición celular Responders vs Non-responders

fig, ax = plt.subplots(figsize=(10, 5))

pivot = prop.pivot(index="celltype", columns="Response", values="proportion").fillna(0)
pivot.plot(kind="bar", ax=ax, colormap="Set2", edgecolor="white", width=0.7)

ax.set_xlabel("Tipo celular", fontsize=12)
ax.set_ylabel("Proporción", fontsize=12)
ax.set_title("Composición celular: Responders vs Non-responders", fontsize=13)
ax.legend(title="Response", bbox_to_anchor=(1.01, 1), loc="upper left")
ax.tick_params(axis="x", rotation=30)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "barplot_celltype_response.png"), dpi=200)
plt.show()
plt.close()
print(f"Barplot guardado en: {FIGURES_DIR}")

Barplot guardado en: /beegfs/home/iruizdealda/HCC_singlecell_project/figures/crossval_AtezoBev


In [11]:
# ── 4. COMPOSICIÓN POR WEEK ──────────────────────────────────
# Ver si la composición celular cambia a lo largo del tiempo
# (W0 → W3 → W6) y si ese cambio es distinto en Responders vs Non-responders

prop_week = (
    adata.obs
    .groupby(["Week", "Response", "celltype"], observed=True)
    .size()
    .reset_index(name="n")
)
prop_week["total"] = prop_week.groupby(["Week", "Response"])["n"].transform("sum")
prop_week["proportion"] = prop_week["n"] / prop_week["total"]

print("Semanas disponibles:", adata.obs["Week"].unique().tolist())
print("\nComposición por Week y Response:")
print(prop_week.pivot_table(
    index="celltype",
    columns=["Week", "Response"],
    values="proportion"
).round(3))

Semanas disponibles: ['W0']

Composición por Week y Response:
Week                        W0                       
Response    DeathBeforeImaging NonResponder Responder
celltype                                             
B                        0.015        0.033     0.045
Endothelial              0.050        0.103     0.067
Fibroblast               0.027        0.041     0.026
Hepatocyte               0.673        0.344     0.461
Myeloid                  0.058        0.115     0.123
T/NK                     0.177        0.364     0.279


/tmp/ipykernel_4115021/3764222641.py:11: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  prop_week["total"] = prop_week.groupby(["Week", "Response"])["n"].transform("sum")
/tmp/ipykernel_4115021/3764222641.py:16: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  print(prop_week.pivot_table(


In [12]:
# Heatmap de composición por Week y Response

pivot_week = prop_week.pivot_table(
    index="celltype",
    columns=["Week", "Response"],
    values="proportion"
).fillna(0)

fig, ax = plt.subplots(figsize=(12, 6))
sns.heatmap(
    pivot_week,
    annot=True,
    fmt=".2f",
    cmap="YlOrRd",
    ax=ax,
    linewidths=0.5
)
ax.set_title("Proporción de tipos celulares por Week y Response", fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, "heatmap_celltype_week_response.png"), dpi=200)
plt.show()
plt.close()
print(f"Heatmap guardado en: {FIGURES_DIR}")

/tmp/ipykernel_4115021/4174445476.py:3: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  pivot_week = prop_week.pivot_table(


Heatmap guardado en: /beegfs/home/iruizdealda/HCC_singlecell_project/figures/crossval_AtezoBev


In [13]:
import pandas as pd
meta = pd.read_csv("/beegfs/home/iruizdealda/HCC_singlecell_project/data/Response_AtezoBev_HCC/meta_data.txt", sep="\t")
print(meta.columns.tolist())
print(meta.head())

['Unnamed: 0', 'Tumour_ID', 'Patient_ID', 'Response', 'Treatment', 'Week']
                    Unnamed: 0 Tumour_ID Patient_ID   Response  Treatment Week
0  AAACCTGCACAGACTT-1_tumour_1  tumour_1      pat24  Responder  atezo+bev   W0
1  AAACCTGCACGAAACG-1_tumour_1  tumour_1      pat24  Responder  atezo+bev   W0
2  AAACCTGTCGGAAATA-1_tumour_1  tumour_1      pat24  Responder  atezo+bev   W0
3  AAACGGGCACATGGGA-1_tumour_1  tumour_1      pat24  Responder  atezo+bev   W0
4  AAACGGGCAGTGGAGT-1_tumour_1  tumour_1      pat24  Responder  atezo+bev   W0
